# Wildfire risk

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset, Subset
import os
import random
import pandas as pd
from PIL import Image
from torchvision import transforms
import math
import matplotlib.pyplot as plt
import numpy as np

import sys
sys.path.append('LoH')
from general_models import *
from experiments.generate_data import *
from utils import *

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

DATASET_DIR = "LoH/experiments/data/wildfire_risk"
RESULTS_DIR = "LoH/experiments/results/wildfire_risk"
IMG_DIR = os.path.join(DATASET_DIR, "images")
CSV_PATH = os.path.join(DATASET_DIR, "labels.csv")


In [ ]:
prepare_wildfire_dataset(
    dataset_dir=DATASET_DIR,
    non_visual_seed=61,
    split_seed=30,
    train_ratio=0.75,
    force_regenerate=False,
)


In [ ]:
visual_variables_names = [
    'dense_forest', 
    'dry_vegetation'
]

non_visual_variable_names = [
    'low_humidity', 
    'strong_wind', 
    'rained_recently', 
    'high_temperature',
    'minimal_human_activity',
    'lightnings_frequent',
    'power_lines_nearby'
]
variable_names = visual_variables_names + non_visual_variable_names

fuel_rule = 'dense_forest | (dry_vegetation & strong_wind)'
dry_rule = 'low_humidity | (high_temperature & ~rained_recently)'
trigger_rule = 'lightnings_frequent | ~minimal_human_activity | power_lines_nearby'

ground_truth_formula = f'({fuel_rule}) & ({dry_rule}) & ({trigger_rule})'


additional_rules = [
    [fuel_rule,
     'dry_vegetation & (strong_wind | low_humidity)',
     'dense_forest',
     'dense_forest & dry_vegetation',
    ],
    [dry_rule,
     'low_humidity & high_temperature & ~rained_recently',
     'high_temperature | (low_humidity & ~rained_recently)',
     '~rained_recently | (low_humidity & strong_wind)',
     '~rained_recently | (low_humidity & high_temperature)',
     'dry_vegetation & ~rained_recently',
    ],
    [trigger_rule,
     'lightnings_frequent & minimal_human_activity',
     'power_lines_nearby & strong_wind',
     'rained_recently & lightnings_frequent',
     '~minimal_human_activity',
    ]
]

parser = LogicalParser(variable_names)
n_non_visual_variables = len(non_visual_variable_names)
n_variables = len(variable_names)

all_rules = []
for rules in additional_rules:
    all_rules.extend(rules)

In [ ]:
# create dataset with persisted visual/non-visual features and shared train/val split
class ImageDataset(Dataset):
    def __init__(self, image_dir, csv_file, visual_variables_names, non_visual_variable_names, transform=None):
        self.image_dir = image_dir
        self.df = pd.read_csv(csv_file)
        self.visual_variables_names = visual_variables_names
        self.non_visual_variable_names = non_visual_variable_names
        self.transform = transform

        required_cols = ['filename', 'wildfire_risk', 'split'] + visual_variables_names + non_visual_variable_names
        missing = [col for col in required_cols if col not in self.df.columns]
        if missing:
            raise ValueError(f"labels.csv missing required columns: {missing}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.image_dir, row['filename'])
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        visual_features = torch.tensor([row[var] for var in self.visual_variables_names], dtype=torch.float32)
        non_visual_features = torch.tensor([row[var] for var in self.non_visual_variable_names], dtype=torch.float32)
        y = torch.tensor([row['wildfire_risk']], dtype=torch.float32)
        return image, visual_features, non_visual_features, y


# normalize images
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

ds_images = ImageDataset(
    IMG_DIR,
    CSV_PATH,
    visual_variables_names,
    non_visual_variable_names,
    transform=transform,
)

split_series = ds_images.df['split'].astype(str).str.lower()
train_indices = np.where(split_series == 'train')[0].tolist()
val_indices = np.where(split_series.isin(['val', 'validation', 'test']))[0].tolist()
if len(train_indices) == 0 or len(val_indices) == 0:
    raise ValueError('Invalid split column. Expected non-empty train and val partitions.')

train_dataset = Subset(ds_images, train_indices)
val_dataset = Subset(ds_images, val_indices)
# Keep backward-compatible names used by later cells.
test_dataset = val_dataset

batch_size = 256
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = val_loader

print(f"Dataset created with {len(ds_images)} samples.")
print(f"Train/Val split: {len(train_dataset)}/{len(val_dataset)}")

for images, visual_feats, non_visual_feats, labels in train_loader:
    print(f"Batch of images shape: {images.shape}")
    print(f"Batch of visual features shape: {visual_feats.shape}")
    print(f"Batch of non-visual features shape: {non_visual_feats.shape}")
    print(f"Batch of labels shape: {labels.shape}")
    break


In [ ]:
def print_wildfire_dataset_stats(visual, non_visual, labels, train_idx, val_idx, visual_names, non_visual_names):
    labels = labels.astype(np.float32).reshape(-1)
    visual = visual.astype(np.float32)
    non_visual = non_visual.astype(np.float32)

    print("Wildfire dataset statistics")
    print(f"- samples: {len(labels)}")
    print(f"- train/val: {len(train_idx)}/{len(val_idx)}")
    print(f"- positive labels: {int(labels.sum())}/{len(labels)} ({labels.mean():.3f})")
    print(f"- train positive ratio: {labels[train_idx].mean():.3f}")
    print(f"- val positive ratio: {labels[val_idx].mean():.3f}")
    print("- visual feature prevalence:")
    for i, name in enumerate(visual_names):
        print(f"  {name}: {visual[:, i].mean():.3f}")
    print("- non-visual feature prevalence:")
    for i, name in enumerate(non_visual_names):
        print(f"  {name}: {non_visual[:, i].mean():.3f}")

visual = ds_images.df[visual_variables_names].to_numpy(dtype=np.float32)
non_visual = ds_images.df[non_visual_variable_names].to_numpy(dtype=np.float32)
labels = ds_images.df['wildfire_risk'].to_numpy(dtype=np.float32).reshape(-1)
train_idx = np.array(train_indices, dtype=np.int32)
val_idx = np.array(val_indices, dtype=np.int32)

print_wildfire_dataset_stats(
    visual,
    non_visual,
    labels,
    train_idx,
    val_idx,
    visual_variables_names,
    non_visual_variable_names,
)


In [ ]:
class SimpleCNN(torch.nn.Module):
    def __init__(self, n_output=1, dropout=0.2):
        super(SimpleCNN, self).__init__()
        self.conv1 = torch.nn.Conv2d(3, 16, kernel_size=5, stride=1, padding=2)
        self.pool = torch.nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = torch.nn.Conv2d(16, 16, kernel_size=5, stride=1, padding=2)
        self.conv3 = torch.nn.Conv2d(16, 16, kernel_size=3, stride=1, padding=1)
        self.dropout_conv = torch.nn.Dropout2d(p=dropout)
        self.fc = torch.nn.Linear(16 * 32 * 32, 32)
        self.fc2 = torch.nn.Linear(32, n_output)
        self.sigmoid = torch.nn.Sigmoid()

    def forward(self, x_image):
        x = self.pool(torch.relu(self.conv1(x_image)))
        x = self.dropout_conv(x)
        x = self.pool(torch.relu(self.conv2(x)))
        x = self.dropout_conv(x)
        x = self.pool(torch.relu(self.conv3(x)))
        x = self.dropout_conv(x)
        x = x.view(-1, 16 * 32 * 32)
        x = torch.relu(self.fc(x))
        x = self.sigmoid(self.fc2(x))
        return x
    

def evaluate_model(cnn, model, data_loader, device, verbose=False):
    model.eval(); cnn.eval()
    with torch.no_grad():
        all_predicted, all_labels, all_predicted_visual_feats, all_labels_visual_feats = [], [], [], []
        for img, visual_feats, non_visual_feats, target in data_loader:
            img, visual_feats, non_visual_feats, target = img.to(device), visual_feats.to(device) + 0., non_visual_feats.float().to(device), target.squeeze(1).to(device) + 0.
            predicted_visual_feats = cnn(img)
            data = torch.cat((predicted_visual_feats, non_visual_feats), dim=1)
            output = model(data)
            predicted = (output >= 0.5).float()
            all_predicted.append(predicted.cpu())
            all_labels.append(target.cpu())
            if verbose:
                all_predicted_visual_feats.append((predicted_visual_feats > 0.5).float().cpu())
                all_labels_visual_feats.append(visual_feats.cpu())
        all_predicted = torch.cat(all_predicted, dim=0).numpy()
        all_labels = torch.cat(all_labels, dim=0).numpy()
        acc, f1 = eval(all_predicted, all_labels)
        mean_visual_list, f1_visual_list = [], []
        if verbose:
            all_predicted_visual_feats = torch.cat(all_predicted_visual_feats, dim=0).numpy()
            all_labels_visual_feats = torch.cat(all_labels_visual_feats, dim=0).numpy()
            for i in range(all_labels_visual_feats.shape[1]):
                acc_visual, f1_visual = eval(all_predicted_visual_feats[:, i], all_labels_visual_feats[:, i])
                mean_visual_list.append(np.mean(all_predicted_visual_feats[:, i])); 
                f1_visual_list.append(f1_visual)
    model.train(); cnn.train()
    return acc, f1, mean_visual_list, f1_visual_list

        
def train_model(cnn, model, train_loader, optimizer, n_epochs, device, lambda_reg=0.0, epoch_start_cnn_update=0,
                val_loader=None, verbose=True, early_stopping=False, early_stopping_patience=16,
                criterion=torch.nn.BCELoss()):
    
    f1_scores_train = []
    f1_scores_val = []  
    model.to(device)
    
    model.train()
    batch = 0
    counter_no_improvement = 0
    last_f1s = [0, 0]
    stopped = False

    for epoch in range(n_epochs):
        if stopped:
            break
        for choose in model.operands:
            choose.gradients = []
        for img, visual_feats, non_visual_feats, target in train_loader:
            img, non_visual_feats, target = img.to(device), non_visual_feats.float().to(device), target.squeeze(1).to(device) + 0.
            
            optimizer.zero_grad()
            visual_feats = cnn(img)
            if epoch < epoch_start_cnn_update:
                visual_feats = visual_feats.detach()
            data = torch.cat((visual_feats, non_visual_feats), dim=1)
            output = model(data)
            loss = criterion(output, target)

            try:
                for choose in model.operands:
                    loss += lambda_reg * choose.w[0] / len(img)
            except:
                pass

            loss.backward()
            if verbose:
                for choose in model.operands:
                    choose.gradients.append(choose.Z.grad[0].item())
                
            optimizer.step()
        
            batch += 1
        print()
        print(f"Epoch {epoch+1}/{n_epochs}, Batch {batch} - Loss: {loss.item():.6f}")
        acc_train, f1_train, mean_visual_train_list, f1_visual_train_list = evaluate_model(cnn, model, train_loader, device, verbose=verbose)
        acc_val, f1_val, mean_visual_val_list, f1_visual_val_list = evaluate_model(cnn, model, val_loader, device, verbose=verbose) if val_loader else (None, None)
        f1_scores_train.append(f1_train)
        f1_scores_val.append(f1_val)
        print(f"\t Train F1: {f1_train:.4f}, Val F1: {f1_val:.4f}")
        if verbose:
            for i, (f1_visual_train, f1_visual_val) in enumerate(zip(f1_visual_train_list, f1_visual_val_list)):
                print(f"\t Visual Feats {visual_variables_names[i]} - Train f1: {f1_visual_train:.4f}, Val f1: {f1_visual_val:.4f}")
                print(f"\t Mean predicted value - Train: {mean_visual_train_list[i]:.4f}, Val: {mean_visual_val_list[i]:.4f}")
            #print(f"Train Loss: {loss.item():.6f}")
            formula = model.extract_formula()
            print("Formula: ", formula)
            simplified_formula = sympy.parse_expr(formula)
            print("Simplified formula: ", simplified_formula)
            
            print("Weights: ")
            for i, choose in enumerate(model.operands):
                print(f"{i}  weight: {choose.get_weights()[0].item():.3f}  gradient: {choose.gradients} \t {choose}")
    
        if f1_val == last_f1s[0] and f1_train == last_f1s[1]:
            counter_no_improvement += 1
        else:
            counter_no_improvement = 0
            last_f1s = [f1_val, f1_train]
        if early_stopping and ((f1_train > 0.999 or f1_val > 0.995) or counter_no_improvement > early_stopping_patience):
            print("Early stopping at epoch ", epoch)
            stopped = True
            break
            
    return f1_scores_train, f1_scores_val


def repeat_experiment(get_model, lr=0.1, lr_cnn=0.0005, dropout=0.2, lambda_reg=0.0, epoch_start_cnn_update=0,
                      n_epochs=30, repetitions=5, verbose=False, early_stopping=False, criterion=torch.nn.BCELoss()):
    
    global train_loader, test_loader
    global device
    
    f1_train = []
    f1_val = []
    models = []

    for i in range(repetitions):
        print(f"Repetition {i+1}/{repetitions}")
        model = get_model()
        cnn = SimpleCNN(n_output=2, dropout=dropout).to(device)
        #init_simplecnn(cnn)
        model.to(device); cnn.to(device)
        optimizer = torch.optim.Adam([{'params': model.parameters(), 'lr': lr},
                                      {'params': cnn.parameters(), 'lr': lr_cnn}])
        f1_scores_train, f1_scores_val = train_model(cnn, model, train_loader, optimizer, n_epochs, device, val_loader=test_loader, 
                                                     lambda_reg=lambda_reg, epoch_start_cnn_update=epoch_start_cnn_update,
                                                     verbose=verbose, early_stopping=early_stopping, criterion=criterion)
        f1_train.append(f1_scores_train)
        f1_val.append(f1_scores_val)
        print()
        print(model.extract_formula())
        print("----------------------------------------------------------")
        print()
        models.append(model)
        
    # fill in the missing values with last value
    max_length = max(len(f1) for f1 in f1_train)
    for i in range(repetitions):
        while len(f1_train[i]) < max_length:
            f1_train[i].append(f1_train[i][-1])
            f1_val[i].append(f1_val[i][-1])

    f1_train = np.array(f1_train)
    f1_val = np.array(f1_val)
        
    return models, f1_train, f1_val


def steps_to_convergence(f1_sequence, tol=1e-3):
    """Return the first evaluation step that reaches the best F1 within `tol`."""
    best_f1 = float(np.max(f1_sequence))
    for idx, value in enumerate(f1_sequence, start=1):
        if best_f1 - float(value) <= tol:
            return idx
    return len(f1_sequence)


def plots(f1_train_hist, f1_val_hist, also_mean=False):
    if also_mean:    
        mean_f1_train = np.mean(f1_train_hist, axis=0)
        mean_f1_val = np.mean(f1_val_hist, axis=0)
        plt.plot(mean_f1_train, label='Train F1')
        plt.plot(mean_f1_val, label='Validation F1')
        plt.xlabel('Epochs')
        plt.ylabel('F1 Score')
        plt.title('Mean F1 Score over Repetitions')
        plt.legend()
        plt.show()

    #plot all
    for i in range(f1_train_hist.shape[0]):
        #plt.plot(f1_train_hist[i], color='blue', alpha=0.3)
        plt.plot(f1_val_hist[i])
    plt.xlabel('Epochs')
    plt.ylabel('F1 Score')
    plt.title('Validation F1 Score over Repetitions')
    plt.show()

def store(models, f1_train_hist, f1_val_hist, name):
    os.makedirs(RESULTS_DIR, exist_ok=True)
    np.save(f'{RESULTS_DIR}/models_{name}.npy', models)
    np.save(f'{RESULTS_DIR}/f1_train_hist_{name}.npy', f1_train_hist)
    np.save(f'{RESULTS_DIR}/f1_val_hist_{name}.npy', f1_val_hist)

In [ ]:
model = lambda: select_one_rule_per_list_architecture(additional_rules, variable_names, conjunction=True, choosedual=True)
print(model())

models, f1_train_hist, f1_val_hist = repeat_experiment(
        model,
        lr = 0.08,
        lr_cnn=8e-4,
        dropout=0.15,
        verbose=False,
        repetitions=20,
        lambda_reg=0.,
        epoch_start_cnn_update=0,
        n_epochs=25
    )

store(models, f1_train_hist, f1_val_hist, 'select_one_per_list')

In [ ]:
plots(f1_train_hist, f1_val_hist)

In [ ]:
model2 = lambda: select_rules_architecture(all_rules, variable_names, conjunction=True, choosedual=True)
print(model2())

models2, f1_train_hist2, f1_val_hist2 = repeat_experiment(
        model2,
        lr = 0.08,
        lr_cnn=8e-4,
        dropout=0.15,
        verbose=False,
        repetitions=20,
        lambda_reg=0.,
        epoch_start_cnn_update=0,
        n_epochs=25
    )

store(models2, f1_train_hist2, f1_val_hist2, 'select_rules_among_all')

In [ ]:
plots(f1_train_hist2, f1_val_hist2)

In [ ]:
#model3 = lambda: And(parser.parse(fuel_rule), select_one_rule_per_list_architecture(additional_rules[1:], variable_names, conjunction=True, choosedual=True))
#print(model3())
#
#models3, f1_train_hist3, f1_val_hist3 = repeat_experiment(
#        model3,
#        lr = 0.08,
#        lr_cnn=8e-4,
#        dropout=0.15,
#        verbose=False,
#        repetitions=20,
#        lambda_reg=0.,
#        epoch_start_cnn_update=1,
#        n_epochs=30
#    )
#
#store(models3, f1_train_hist3, f1_val_hist3, 'fuel_given_select_one_per_other_list2')

In [ ]:
#plots(f1_train_hist3, f1_val_hist3)

In [ ]:
model4 = lambda: And(parser.parse(fuel_rule), select_rules_architecture(additional_rules[1] + additional_rules[2], variable_names, conjunction=True, choosedual=True))
print(model4())

models4, f1_train_hist4, f1_val_hist4 = repeat_experiment(
        model4,
        lr = 0.08,
        lr_cnn=8e-4,
        dropout=0.15,
        verbose=False,
        repetitions=20,
        lambda_reg=0.,
        epoch_start_cnn_update=0,
        n_epochs=25

    )

store(models4, f1_train_hist4, f1_val_hist4, 'fuel_given_select_rules_among_others')

In [ ]:
plots(f1_train_hist4, f1_val_hist4)

In [29]:
model_ground_truth = lambda: parser.parse(ground_truth_formula)
print(model_ground_truth())

models5, f1_train_hist5, f1_val_hist5 = repeat_experiment(
        model_ground_truth,
        lr = 0.08,
        lr_cnn=8e-4,
        dropout=0.15,
        verbose=False,
        repetitions=20,
        lambda_reg=0.,
        epoch_start_cnn_update=0,
        n_epochs=25
    )

store(models5, f1_train_hist5, f1_val_hist5, 'ground_truth_given2')

((dense_forest | (dry_vegetation & strong_wind)) & (low_humidity | (high_temperature & ~ rained_recently)) & (lightnings_frequent | ~ minimal_human_activity | power_lines_nearby))
Repetition 1/20

Epoch 1/25, Batch 6 - Loss: 0.322408
	 Train F1: 0.8401, Val F1: 0.8643

Epoch 2/25, Batch 12 - Loss: 0.235809
	 Train F1: 0.9601, Val F1: 0.9638

Epoch 3/25, Batch 18 - Loss: 0.097589
	 Train F1: 1.0000, Val F1: 1.0000

Epoch 4/25, Batch 24 - Loss: 0.038265
	 Train F1: 0.9985, Val F1: 1.0000

Epoch 5/25, Batch 30 - Loss: 0.031146
	 Train F1: 1.0000, Val F1: 0.9978

Epoch 6/25, Batch 36 - Loss: 0.034262
	 Train F1: 1.0000, Val F1: 1.0000

Epoch 7/25, Batch 42 - Loss: 0.040836
	 Train F1: 1.0000, Val F1: 0.9957

Epoch 8/25, Batch 48 - Loss: 0.036191
	 Train F1: 0.9993, Val F1: 0.9956

Epoch 9/25, Batch 54 - Loss: 0.035908
	 Train F1: 1.0000, Val F1: 0.9935

Epoch 10/25, Batch 60 - Loss: 0.006616
	 Train F1: 0.9993, Val F1: 0.9956

Epoch 11/25, Batch 66 - Loss: 0.007650
	 Train F1: 1.0000, Val 

In [ ]:
plots(f1_train_hist5, f1_val_hist5)

In [ ]:
mean_f1_val = np.mean(f1_val_hist, axis=0)
mean_f1_val2 = np.mean(f1_val_hist2, axis=0)
#mean_f1_val3 = np.mean(f1_val_hist3, axis=0)
mean_f1_val4 = np.mean(f1_val_hist4, axis=0)
mean_f1_val5 = np.mean(f1_val_hist5, axis=0)

plt.plot(mean_f1_val5, label='Full Knowledge')
plt.plot(mean_f1_val2, label='Selecting Reliable Rules')
plt.plot(mean_f1_val, label='Selecting One Rule per Set')
#plt.plot(mean_f1_val3, label='fuel given, one per other list')
plt.plot(mean_f1_val4, label='Partial Knowledge Base (Fuel given)')

plt.xlabel('Epochs')
plt.ylabel('F1 Score')
plt.legend()
plt.show()